In [13]:
import pandas as pd
import numpy as np
import random

def weighted_choice(options):
    r = random.random()
    cumulative = 0
    for value, weight in options:
        cumulative += weight
        if r <= cumulative:
            return value

def rand_range(low, high):
    return random.randint(low, high)

def process_creatures(df):
    # Filter Stage 1
    df = df[df["Stage"] == "Stage 1"].copy()

    # Create new columns
    df["move1_level"] = 1

    df["move2_level"] = df.apply(
        lambda _: weighted_choice([(2, 0.25), (3, 0.50), (4, 0.25)]),
        axis=1
    )
    df["move3_level"] = df.apply(
        lambda _: weighted_choice([(5, 0.25), (6, 0.50), (7, 0.25)]),
        axis=1
    )
    df["move4_level"] = df.apply(
        lambda _: weighted_choice([(8, 0.50), (9, 0.30), (10, 0.20)]),
        axis=1
    )

    # Randomly choose Buff move
    df["buff_move"] = df.apply(
        lambda _: random.choice(["move2","move3","move4"]),
        axis=1
    )

    # Prepare move columns
    df["move1"] = None
    df["move2"] = None
    df["move3"] = None
    df["move4"] = None

    # Assign move powers
    for idx, row in df.iterrows():
        rarity = int(row["Rarity"])

        # Define ranges
        if rarity >= 4:
            ranges = {
                "move1": (28, 45),
                "move2": (40, 62),
                "move3": (50, 75),
                "move4": (60, 90)
            }
        else:
            ranges = {
                "move1": (20, 33),
                "move2": (40, 54),
                "move3": (55, 68),
                "move4": (60, 78)
            }

        # Assign Buff
        buff = row["buff_move"]
        moves = {}

        for m in ["move1", "move2", "move3", "move4"]:
            if m == buff:
                moves[m] = "Buff"
            else:
                low, high = ranges[m]
                moves[m] = rand_range(low, high)

        # Enforce ordering move4 > move3 > move2 > move1
        numeric_moves = {m: v for m, v in moves.items() if v != "Buff"}
        sorted_values = sorted(numeric_moves.values())

        # Reassign sorted values in correct order
        ordered_keys = ["move1", "move2", "move3", "move4"]
        sorted_idx = 0
        for m in ordered_keys:
            if moves[m] != "Buff":
                moves[m] = sorted_values[sorted_idx]
                sorted_idx += 1

        # Save back
        df.at[idx, "move1"] = moves["move1"]
        df.at[idx, "move2"] = moves["move2"]
        df.at[idx, "move3"] = moves["move3"]
        df.at[idx, "move4"] = moves["move4"]

    return df


In [14]:
df = pd.read_csv("Temporal Rift.csv")
result = process_creatures(df)
print(result)


           id_output            Name    Stage       Type               image  \
0   Temporal Rift_01         Toxikin  Stage 1     Mystic         Toxikin.png   
3   Temporal Rift_04        Mythloon  Stage 1     Mystic        Mythloon.png   
5   Temporal Rift_06         Echolet  Stage 1     Mystic         Echolet.png   
7   Temporal Rift_08         Orkidia  Stage 1     Mystic         Orkidia.png   
9   Temporal Rift_10      Umbropplet  Stage 1     Mystic      Umbropplet.png   
11  Temporal Rift_12         Lumimba  Stage 1     Mystic         Lumimba.png   
14  Temporal Rift_15         Nyamura  Stage 1     Mystic         Nyamura.png   
16  Temporal Rift_17  Faygoal Mystic  Stage 1     Mystic  Faygoal Mystic.png   
17  Temporal Rift_18        Pebblift  Stage 1       Wind        Pebblift.png   
19  Temporal Rift_20        Loonaqua  Stage 1       Wind        Loonaqua.png   
22  Temporal Rift_23       Nimbearis  Stage 1       Wind       Nimbearis.png   
23  Temporal Rift_24      Cycloblaze  St

In [15]:
new_df=df.merge(result, left_on="id_output", right_on="id_output", how="left")

In [16]:
new_df["move1 name"] = None 
new_df["move2 name"] = None 
new_df["move3 name"] = None 
new_df["move4 name"] = None 

new_df["move1 buff"] = None 
new_df["move2 buff"] = None 
new_df["move3 buff"] = None 
new_df["move4 buff"] = None 

new_df["move1 buff p"] = None 
new_df["move2 buff p"] = None 
new_df["move3 buff p"] = None 
new_df["move4 buff p"] = None 

In [17]:
cols = [
    "id_output",
    "move1 name",
    "move1",
    "move1_level",
    "move1 buff",
    "move1 buff p",
    "move2 name",
    "move2",
    "move2_level",
    "move2 buff",
    "move2 buff p",
    "move3 name",
    "move3",
    "move3_level",
    "move3 buff",
    "move3 buff p",
    "move4 name",
    "move4",
    "move4_level",
    "move4 buff",
    "move4 buff p",
]

filtered_df = new_df[cols]

In [18]:
filtered_df.merge(result, left_on="id_output", right_on="id_output", how="left").to_csv("Tcg game - with moves.csv", index=False)
